# scRNA-seq Analysis Pipeline — Reference Documentation

> **How to use this notebook:** This is a methods reference and decision guide, not a script to run top-to-bottom. Each section explains *what problem a step solves*, *which tools are available*, and *when to choose each option*. For the actual implementation with the Yang et al. study parameters, see the numbered step notebooks (Steps 1–7).

---

## Why single-cell RNA-seq, and why is it hard?

Traditional "bulk" RNA-seq measures average gene expression across all cells in a tissue. It tells you that a gene goes up in obese mice — but not *which cells* are responsible. A fat tissue is not one thing: it contains fat cells (adipocytes), immune cells (macrophages, T cells, B cells), stem cells (adipose stem cells), blood vessel cells (endothelial), and more. A bulk measurement averages across all of them.

Single-cell RNA-seq (scRNA-seq) solves this by sequencing each cell individually. Every row in the output matrix is one cell. This lets you ask: *was the change driven by the fat cells, or by the immune cells, or by a rare stem cell population?* The Yang et al. paper used this resolution to discover that **mesenchymal stem cells (MSCs) are the primary responders to both obesity and exercise** — a finding that was completely invisible to bulk methods.

The catch: measuring individual cells introduces noise and artifacts that bulk RNA-seq does not have. This is what the pipeline exists to address.

## The single-cell measurement process (for context)

1. Tissue is dissociated into a suspension of individual cells
2. Cells are loaded into a 10x Genomics microfluidic chip; each cell is captured in its own oil droplet with a uniquely barcoded gel bead
3. Cell RNA is reverse-transcribed into cDNA, tagged with the barcode + a unique molecular identifier (UMI) per transcript
4. All cDNA is pooled and sequenced together
5. CellRanger demultiplexes by barcode → per-cell count matrix

This process introduces three categories of artifact that the pipeline corrects:
- **Ambient RNA** from lysed cells contaminating all droplets (Step 3, SoupX)
- **Low-quality cells** (dead, fragmented, empty droplets) that passed the cell detector (Steps 2–3)
- **Doublets** — two cells captured in one droplet (Step 3, Scrublet)

## The full pipeline structure

```
[Step 1]  CellRanger metrics ──── Audit sample quality before the pipeline runs
[Step 2]  Diagnostic plots ─────── Visualize distributions to set filtering thresholds
[Step 3]  Per-sample processing ── Ambient RNA removal → QC filter → doublet detection
[Step 4]  Pseudobulk clustering ── Sanity check: do samples group by biology or batch?
[Step 5]  Integration ───────────── Merge 39 samples → 204,883 cells → tSNE/UMAP/clusters
[Step 6]  Add metadata ───────────── Join experimental conditions (diet, exercise) onto cells
[Step 7]  Downstream analysis ────── Cell type annotation, DEGs, visualization
[Bulk]    Bulk DESeq2 ─────────────── Independent validation: 1,386 tissue-level DEGs
```

This reference document covers the decision points at each stage, the Python tools available, and the biological rationale for key choices.

---
# Part 1 — Per-Sample Processing

## 1.1 Ambient RNA Removal

When cells are dissociated into suspension, some lyse and release their RNA into the surrounding medium. Every droplet captures a mixture of cell-derived RNA and this ambient "soup". If uncorrected, highly expressed genes from abundant cell types (e.g., hemoglobin from red blood cells, immunoglobulins from B cells) appear to be expressed in all cell types, creating false signals.

**Three estimation modes**, in order of increasing biological specificity:

1. **Auto** — uses the expression patterns across all cells and empty droplets to estimate the contamination fraction automatically via expectation-maximization. Works well when contamination is moderate and the dataset is diverse.

2. **Fixed rate** — applies a user-specified contamination fraction (e.g., 10%, 20%) uniformly to all cells. Useful for sensitivity analysis or when the auto-estimate is unstable. The Yang et al. study used a fixed 20% rate for adipose and muscle, which have high ambient RNA due to tissue fragility during dissociation.

3. **Manual marker genes** — uses known tissue-specific genes that should be unexpressed in most cell types (hemoglobin genes for non-RBC cells, immunoglobulin genes for non-B cells). Any expression of these genes is attributed to ambient contamination, providing a direct calibration signal.

**Rule of thumb:** after applying correction, re-plot hemoglobin gene expression on the UMAP. If it is still broadly expressed, increase the correction rate or switch to manual mode.

**Python tool:** Implemented manually in `sample_level_processing.ipynb` (Step 3). Alternatives include `CellBender` (neural network approach, run as a command-line tool before loading data into Python) or calling SoupX via `rpy2`.

In [ ]:
import numpy as np
import pandas as pd
import scanpy as sc
import scipy.sparse as sp
from pathlib import Path

# ── Ambient RNA removal (see sample_level_processing.ipynb for full implementation) ──

# Load raw (all barcodes) and filtered (cells only) matrices from CellRanger
# adata_raw  = sc.read_10x_mtx("outs/raw_feature_bc_matrix/")
# adata_filt = sc.read_10x_mtx("outs/filtered_feature_bc_matrix/")

# Empty droplets = barcodes in raw but not filtered
# soup_profile = estimate ambient RNA composition from empty droplets

# Mode 1: auto — estimate rho from low-expression genes
# Mode 2: fixed — rho = 0.10 / 0.15 / 0.20
# Mode 3: manual — calibrate using hemoglobin / immunoglobulin genes
hb_genes = ["Hbb-bt", "Hbb-bs", "Hbb-bh2", "Hbb-bh1", "Hbb-y",
            "Hba-x", "Hba-a1", "Hba-a2"]
ig_genes  = ["Igha", "Ighe", "Ighg2c", "Ighg2b", "Ighg1",
             "Ighg3", "Ighd", "Ighm", "Ighj4", "Ighj3", "Ighj2", "Ighj1"]
non_expressing_genes = hb_genes + ig_genes

# Correction: subtract estimated ambient contribution per cell
# X_corrected = round(max(X - rho * total_counts * soup_profile, 0))

## 1.2 Filtering Out Low-Quality Cells

Four criteria identify low-quality barcodes:

1. **Gene count lower bound** (`nFeature_RNA > 200`) — removes empty droplets and cell debris, which have very few detected genes.
2. **Gene count upper bound** (`nFeature_RNA < 6000`) — removes putative doublets, which have abnormally high gene diversity from two cell transcriptomes merged.
3. **UMI count lower bound** (`nCount_RNA > 500`) — removes very shallow barcodes unlikely to represent real cells.
4. **Mitochondrial fraction upper bound** (`percent_mt < 10–30%` depending on tissue) — removes dying or lysed cells. The appropriate threshold is tissue-dependent: metabolically active tissues (muscle, brown fat) have higher baseline mitochondrial content.
5. **Proliferating cells** (optional, `Mki67 == 0`) — removes actively cycling cells whose transcriptome is dominated by cell cycle genes, which would cause them to cluster by cell cycle phase rather than identity.

The thresholds should be set per-sample based on the diagnostic plots from Step 2, not applied globally.

In [ ]:
# ── QC metric calculation and cell filtering ──

# adata.var["mt"] = adata.var_names.str.startswith("mt-")  # mouse
# sc.pp.calculate_qc_metrics(adata, qc_vars=["mt"], inplace=True)

# Apply thresholds (adjust per-sample based on Step 2 diagnostic plots)
nfeature_low  = 200
nfeature_high = 6000
ncount_low    = 500
per_mt_high   = 10

# keep = (
#     (adata.obs["n_genes_by_counts"]  > nfeature_low)  &
#     (adata.obs["n_genes_by_counts"]  < nfeature_high) &
#     (adata.obs["total_counts"]        > ncount_low)    &
#     (adata.obs["pct_counts_mt"]       < per_mt_high)
# )
# adata = adata[keep].copy()

# Optional: remove proliferating cells
# marker_idx = adata.var_names.get_loc("Mki67")
# mki67_expr = adata.X[:, marker_idx].toarray().flatten()
# adata = adata[mki67_expr == 0].copy()

print("Cell filtering parameters set. See sample_level_processing.ipynb for execution.")

## 1.3 Doublet Removal

Doublets (two cells captured in one droplet) create artificial hybrid transcriptomes. They can appear as spurious cell types between two real populations or inflate cluster sizes.

**DoubletFinder / Scrublet approach:**
1. Simulate artificial doublets by summing pairs of real cell count vectors
2. Embed real + simulated cells jointly in PCA space
3. Score each real cell by the fraction of its k nearest neighbors that are simulated doublets
4. Classify the top-scoring cells as doublets, where the expected number = `0.031 × n_cells` (10x Genomics published rate for ~4,000 cells, scaling with cell count)

**Homotypic correction:** Two cells of the same type are harder to detect as doublets (their merged transcriptome looks like a deeper version of the single-cell transcriptome). The expected doublet count is adjusted downward by the estimated fraction of doublets that are homotypic, derived from cluster proportions.

**Python tool:** `scrublet` (see `sample_level_processing.ipynb` Step 3).

In [ ]:
# ── Doublet detection with Scrublet ──
# import scrublet as scr

# expected_doublet_rate = 0.031   # 3.1% for ~4,000 cells; scales with cell count
# scrub = scr.Scrublet(adata.X, expected_doublet_rate=expected_doublet_rate)
# doublet_scores, predicted_doublets = scrub.scrub_doublets()
# adata.obs["doublet_score"]     = doublet_scores
# adata.obs["predicted_doublet"] = predicted_doublets
# adata = adata[~predicted_doublets].copy()

print("Doublet detection shown as reference. See sample_level_processing.ipynb for execution.")

---
# Part 2 — All Samples Combined

## 2.1 Sample Filtering Based on Pseudobulk Profiles

Before integration, verify that samples cluster by biology (tissue, condition) rather than by technical artifacts. Aggregate per-cell expression to a single vector per sample (mean across cells), compute Spearman correlation between all sample pairs, and cluster hierarchically.

A sample that clusters with the wrong tissue group, or is an outlier within its group, should be investigated before integration — it may represent a labeling error, sample swap, or failed library.

See `sample_level_pseudobulk_clustering.ipynb` (Step 4) for the full implementation.

In [ ]:
# ── Pseudobulk per-sample mean expression ──
# For each sample object:
#   mean_expr = np.array(adata.X.mean(axis=0)).flatten()
#   pseudobulk[sample_name] = pd.Series(mean_expr, index=adata.var_names)

# bulk_df = pd.DataFrame(pseudobulk)            # genes x samples
# corr    = bulk_df.corr(method="spearman")      # Spearman correlation
# import seaborn as sns
# sns.clustermap(corr, method="ward", cmap="Blues")

print("Pseudobulk clustering shown as reference. See sample_level_pseudobulk_clustering.ipynb.")

## 2.2 Merge Samples and Embed

Refer to Kobak & Berens (2019, *Nature Communications*) "The art of using t-SNE for single-cell transcriptomics" for best practices. Key recommendations implemented in `fancy_tsne.R` / `sample_integration.ipynb`:

- **Depth normalization:** normalize to the per-sample median UMI count (not 10,000) to avoid distorting relative expression after log transformation
- **PCA:** fixed 50 components
- **tSNE (FIt-SNE):**
  - Initialize from PCA (divide PC1–2 by `sd(PC1)`, multiply by 0.0001)
  - Learning rate = `n / 12`
  - Perplexity = 30 for very large datasets (> 100,000 cells); combine with `n/100` for smaller datasets
  - Exaggeration factor ≈ 4 for very large datasets
- **Note:** tSNE results can be rotated and flipped without changing inter-point distances, but must not be stretched horizontally or vertically

In [ ]:
# ── Merge + embed (see sample_integration.ipynb for full implementation) ──
# import anndata as ad
# from openTSNE import TSNE as openTSNE

# combined = ad.concat(adatas, join="inner")
# sc.pp.normalize_total(combined, target_sum=median_count)
# sc.pp.log1p(combined)
# sc.pp.highly_variable_genes(combined, n_top_genes=2000)
# sc.tl.pca(combined, n_comps=50)

# n = combined.n_obs
# pca_init = combined.obsm["X_pca"][:, :2] / combined.obsm["X_pca"][:, 0].std() * 0.0001
# tsne = openTSNE(perplexity=[30, n//100], initialization=pca_init,
#                 learning_rate=n/12, n_iter=1000, n_jobs=-1)
# combined.obsm["X_tsne"] = np.array(tsne.fit(combined.obsm["X_pca"]))

# sc.pp.neighbors(combined, n_pcs=50)
# sc.tl.umap(combined)
# sc.tl.leiden(combined, resolution=0.4)

print("Merge + embed shown as reference. See sample_integration.ipynb.")

## 2.3 Batch Effect Assessment and Correction

Plot each sample's cells on the UMAP colored by sample ID. If cells from different samples of the same type intermix freely, batch effects are negligible. If samples form discrete islands by sample ID (rather than by cell type), batch correction is needed.

Useful resources:
- [Understanding batch effects in scRNA-seq](https://constantamateur.github.io/2020-06-09-scBatch1/)
- [Removing batch effects](https://constantamateur.github.io/2020-10-24-scBatch2/)

**Available Python tools:**

| Tool | Approach | Python package |
|------|----------|---------------|
| Harmony | Iterative PCA embedding correction | `harmonypy` |
| Seurat rPCA | Anchor-based PCA alignment | `rpy2` + R Seurat |
| LIGER | Matrix factorization | `pyliger` |
| scVI | Variational autoencoder | `scvi-tools` |
| scANVI | scVI + cell type labels | `scvi-tools` |

In [ ]:
# ── Batch correction options ──

# Option 1: Harmony (fastest, works on PCA embedding)
# import harmonypy as hm
# ho = hm.run_harmony(combined.obsm["X_pca"], combined.obs, "sample_ID")
# combined.obsm["X_pca_harmony"] = ho.Z_corr.T
# sc.pp.neighbors(combined, use_rep="X_pca_harmony")

# Option 2: scVI (most powerful, requires training time)
# import scvi
# scvi.model.SCVI.setup_anndata(combined, layer="counts", batch_key="sample_ID")
# model = scvi.model.SCVI(combined)
# model.train()
# combined.obsm["X_scvi"] = model.get_latent_representation()

# Option 3: LIGER (NMF-based, good for non-linear batch effects)
# import pyliger

print("Batch correction options shown as reference.")

---
# Part 3 — Cell Type Annotation

Cell type annotation assigns biological identities to the clusters produced by Leiden/Louvain clustering. A combination of automatic and manual approaches is recommended — automated tools give a rough first pass, manual review with marker genes refines and corrects it.

## 3.1 Automated Annotation

Automated tools annotate cells by comparing their expression profiles to reference datasets:

| R Tool | Python Equivalent | Reference Type | Notes |
|--------|------------------|----------------|-------|
| SciBet | `celltypist` | scRNA-seq references | Probabilistic; good for well-characterized tissues |
| SingleR | `celltypist` | Bulk / microarray references | Works even without scRNA-seq references |
| — | `scANVI` | scRNA-seq (label transfer) | Best when a labeled reference atlas exists |

None of these tools is perfect — they give a rough starting point that requires manual verification. They are most reliable when a high-quality reference for the same tissue and species is available.

## 3.2 Manual Annotation with Marker Genes

Cross-reference automated labels with canonical marker gene expression using dot plots and feature plots. Key markers for the tissues in this study:

| Cell type | Key markers |
|-----------|------------|
| Adipocyte | Adipoq, Fabp4, Lep |
| ASC / Adipocyte precursor | Pdgfra, Ly6a (Sca-1), Cd34 |
| FAP | Pdgfra, Ly6a-, Col3a1 |
| Macrophage | Adgre1 (F4/80), Csf1r, Cd68 |
| T cell | Cd3e, Cd3d, Trac |
| B cell | Cd79a, Ms4a1 (Cd20) |
| NK cell | Nkg7, Klrb1c |
| Endothelial | Pecam1 (Cd31), Cdh5 |
| Pericyte | Rgs5, Pdgfrb |
| Muscle fiber | Myh1, Myh2, Myh4 (fast); Myh7 (slow) |
| Satellite cell | Pax7, Myod1 |

Sub-clustering a broad cluster (e.g., "macrophage") often reveals sub-populations (tissue-resident vs. recruited, M1/M2 polarization states) that are biologically meaningful.

In [ ]:
# ── Automated annotation with CellTypist ──
# pip install celltypist
# import celltypist
# from celltypist import models

# Download a pre-trained model (e.g., for mouse tissues)
# model = models.Model.load(model="Mouse_Immune_Atlas.pkl")
# predictions = celltypist.annotate(adata, model=model, majority_voting=True)
# adata = predictions.to_adata()

# ── Dot plot for manual marker verification ──
# sc.pl.dotplot(
#     adata,
#     var_names=["Adipoq", "Pdgfra", "Adgre1", "Cd3e", "Cd79a", "Pecam1", "Myh1"],
#     groupby="leiden",
#     standard_scale="var",
# )

# ── Cross-reference hierarchical + density clustering ──
# Leiden at multiple resolutions + DBSCAN on UMAP helps disambiguate
# clusters that merge/split at different granularities
# from sklearn.cluster import DBSCAN
# dbscan_labels = DBSCAN(eps=0.5).fit_predict(adata.obsm["X_umap"])

print("Cell type annotation tools shown as reference.")

---
# Part 4 — Downstream Analysis

## 4.1 Differentially Expressed Genes (DEGs)

Choosing the right DE method for scRNA-seq is non-trivial and depends on the experimental design:

| Scenario | Recommended method | Python tool |
|----------|--------------------|-------------|
| Simple within-dataset comparison (mouse, homogeneous samples) | Wilcoxon rank-sum or logistic regression | `sc.tl.rank_genes_groups` |
| Multi-sample comparison with replicates | Pseudobulk + DESeq2 | `pydeseq2` |
| Human samples or high inter-individual variation | Mixed effect models (NEBULA) | `diffxpy`, or `pydeseq2` on pseudobulk |
| Compositional analysis (cell type proportions) | Dirichlet regression | `diffxpy`, `pertpy` |

**Why pseudobulk for multi-sample comparisons?** 
Treating individual cells as independent replicates inflates the effective sample size enormously (10,000 cells from 4 mice ≠ 10,000 independent observations). This produces extremely small p-values even for biologically trivial differences. Pseudobulk aggregation (sum or mean counts per sample per cell type) correctly treats the animal as the statistical unit.

Relevant reading: [Single-cell DE analysis blog post](https://constantamateur.github.io/2020-04-10-scDE/)

In [ ]:
# ── Option 1: Wilcoxon (within-dataset, no replicates) ──
# sc.tl.rank_genes_groups(adata, groupby="cell_type", method="wilcoxon")
# sc.pl.rank_genes_groups_dotplot(adata, n_genes=5)

# ── Option 2: Pseudobulk DESeq2 (multi-sample, recommended) ──
# pip install pydeseq2
# from pydeseq2.dds import DeseqDataSet
# from pydeseq2.ds import DeseqStats

# Step 1: aggregate counts per sample per cell type
# pseudobulk_counts = (
#     pd.DataFrame(
#         adata[adata.obs["cell_type"] == "Macrophage"].layers["counts"].toarray(),
#         index=adata[adata.obs["cell_type"] == "Macrophage"].obs_names,
#         columns=adata.var_names,
#     )
#     .join(adata.obs[["sample_ID", "condition"]])
#     .groupby("sample_ID").sum()
# )

# Step 2: run DESeq2
# dds = DeseqDataSet(counts=pseudobulk_counts, metadata=sample_metadata,
#                   design_factors="condition")
# dds.deseq2()
# stats = DeseqStats(dds, contrast=("condition", "Training", "Sedentary"))
# stats.summary()

# ── Option 3: NEBULA (mixed effects, for human / high-variance data) ──
# No direct Python port — use rpy2 or pseudobulk as approximation

print("DEG methods shown as reference.")

## 4.2 Pathway Enrichment

Pathway enrichment asks whether the DEGs are over-represented in known biological pathways, providing mechanistic interpretation.

Common approaches:
- **ORA (over-representation analysis)** — hypergeometric test on the significant DEG list against pathway gene sets (MSigDB, KEGG, GO)
- **GSEA (gene set enrichment analysis)** — uses the full ranked DEG list, more sensitive than ORA
- **Decoupler** — framework supporting multiple enrichment methods on single-cell data

In [ ]:
# ── Pathway enrichment options ──

# Option 1: gseapy (ORA + GSEA)
# pip install gseapy
# import gseapy as gp
# enr = gp.enrichr(gene_list=deg_list, gene_sets="KEGG_2021_Human", organism="Mouse")
# gp.prerank(rnk=ranked_genes, gene_sets="MSigDB_Hallmark_2020")

# Option 2: decoupler (multiple methods, integrates with AnnData)
# pip install decoupler
# import decoupler as dc
# net = dc.get_progeny(organism="mouse")
# dc.run_mlm(mat=adata, net=net, source="source", target="target", weight="weight")

print("Pathway enrichment tools shown as reference.")

## 4.3 Single-Cell CNV Estimation

Copy number variation (CNV) inference from scRNA-seq is used primarily to identify malignant cells in tumor samples. CNV tools compare expression profiles of putative tumor cells against a normal reference, identifying chromosomal gains and losses from expression imbalances across genomic windows.

Less commonly used in metabolic studies like this one, but relevant if immune cell activation states or clonal dynamics are of interest.

References:
- [inferCNV](https://github.com/broadinstitute/inferCNV/wiki)
- [CopyKAT](https://www.nature.com/articles/s41587-020-00795-2)

In [ ]:
# ── CNV inference ──

# Option: infercnvpy (Python port of inferCNV)
# pip install infercnvpy
# import infercnvpy as cnv
# cnv.io.genomic_position_from_gtf("genes.gtf", adata)
# cnv.tl.infercnv(adata, reference_key="cell_type", reference_cat=["T", "B"])
# cnv.pl.chromosome_heatmap(adata)

print("CNV estimation tools shown as reference.")

## 4.4 Cell-Cell Communication

Cell-cell communication (CCC) tools infer ligand-receptor interactions between cell types from their co-expression patterns. The underlying assumption is that if cell type A expresses a ligand and cell type B expresses the corresponding receptor, they may be communicating via that signaling axis.

In metabolic studies, CCC is particularly relevant for:
- Adipose-immune crosstalk (e.g., adipokine signaling to macrophages)
- Muscle-adipose communication (myokines)
- Paracrine signaling within the stromal vascular fraction

[CellPhoneDB](https://github.com/Teichlab/cellphonedb) is gene co-expression based and has a Python implementation.

In [ ]:
# ── Cell-cell communication ──

# Option 1: CellPhoneDB (ligand-receptor co-expression)
# pip install cellphonedb
# from cellphonedb.src.pipeline.pipeline import call_cellphonedb

# Option 2: LIANA (benchmarked framework, wraps multiple CCC methods)
# pip install liana
# import liana
# liana.mt.rank_aggregate(adata, groupby="cell_type")

# Option 3: CellChat (via rpy2)
# CellChat incorporates signaling pathway databases and network analysis

print("Cell-cell communication tools shown as reference.")

## 4.5 Gene Regulatory Network Analysis

Gene regulatory network (GRN) analysis infers which transcription factors (TFs) are driving the transcriptional programs observed in each cell type or condition. SCENIC (Single-Cell rEgulatory Network Inference and Clustering) is the standard tool, combining co-expression analysis with TF binding motif enrichment to identify regulons — TFs and their predicted target genes.

In exercise/obesity studies, GRN analysis can identify master regulators of adipogenesis, lipid metabolism, or muscle fiber type switching that are activated or repressed by diet or exercise training.

[SCENIC GitHub](https://github.com/aertslab/SCENIC) | [pySCENIC](https://pyscenic.readthedocs.io/)

In [ ]:
# ── Gene regulatory network analysis with pySCENIC ──

# pySCENIC runs in three stages:
# 1. GRN inference: identify co-expression modules (TF + target genes)
# 2. Motif enrichment: filter modules by TF binding motif support
# 3. AUC scoring: score each cell for each regulon's activity

# pip install pyscenic
# from pyscenic.grn import grnboost2
# from pyscenic.ctx import prune2df, df2regulons
# from pyscenic.aucell import aucell

# Stage 1
# ex_matrix = pd.DataFrame(adata.X.toarray(), index=adata.obs_names,
#                          columns=adata.var_names)
# adjacencies = grnboost2(ex_matrix, tf_names=tf_list)

# Stage 2 (requires ranking databases from cisTarget)
# df = prune2df(adjacencies, dbs=[mm10_500bp_db, mm10_10kb_db],
#              motif_annotations=motif_annotations)
# regulons = df2regulons(df)

# Stage 3
# auc_mtx = aucell(ex_matrix, regulons)
# adata.obsm["X_scenic"] = auc_mtx.values

print("pySCENIC workflow shown as reference.")